# DisasterLens BRIGHT training on Kaggle

This notebook resumes the DisasterLens implementation on the attached official BRIGHT Kaggle Dataset. By default it runs M3 after a completed M1–M2 run; set DISASTERLENS_THROUGH_MILESTONE=M4 only after reviewing the M3 reports. Environment variables can extend the same resumable pipeline through M8. It never creates or substitutes training data. Kaggle GPU execution must report an NVIDIA Tesla T4.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import zipfile
import subprocess
import sys
import time

import torch

if not torch.cuda.is_available():
    raise RuntimeError('Kaggle GPU is unavailable. Set accelerator: gpu and rerun; no CPU fallback is allowed.')
gpu_name = torch.cuda.get_device_name(0)
print(f'[Kaggle] GPU: {gpu_name}', flush=True)
if 'T4' not in gpu_name.upper():
    raise RuntimeError(f'Tesla T4 required for this run, but Kaggle assigned: {gpu_name}')

candidates = [Path('/kaggle/working'), Path('/kaggle/input/disaster-lens')]
repo_dir = next((path for path in candidates if (path / 'pyproject.toml').is_file()), None)
if repo_dir is None:
    repo_dir = Path('/kaggle/working/disaster-lens')
    if not (repo_dir / 'pyproject.toml').is_file():
        print('[Kaggle] project files were not uploaded; cloning the public repository', flush=True)
        subprocess.run([
            'git', 'clone', '--depth', '1', '--branch', 'main',
            'https://github.com/kushc2004/disaster-lens.git', str(repo_dir),
        ], check=True)
if not (repo_dir / 'pyproject.toml').is_file():
    raise RuntimeError(f'Repository checkout is incomplete: missing pyproject.toml under {repo_dir}')
repo_dir = repo_dir.resolve()
# Kaggle may mount an attached dataset under a versioned/name-normalized folder.
dataset_candidates = [Path('/kaggle/input/bright-dataset'), Path('/kaggle/input/bright')]
if Path('/kaggle/input').is_dir():
    dataset_candidates.extend(sorted(path for path in Path('/kaggle/input').iterdir() if path.is_dir() and path not in dataset_candidates))
dataset_candidates = list(dict.fromkeys(dataset_candidates))
print('[data] input mounts: ' + ', '.join(str(path) for path in dataset_candidates if path.exists()), flush=True)
def find_bright_root(base):
    suffixes = {'pre-event': '_pre_disaster.tif', 'post-event': '_post_disaster.tif', 'target': '_building_damage.tif'}
    for root in [base, *base.rglob('*')]:
        if root.is_dir() and all((root / name).is_dir() for name in suffixes):
            if all(any((root / name).glob(f'*{suffix}')) for name, suffix in suffixes.items()):
                return root.resolve()
    return None
dataset_root = None
for candidate in dataset_candidates:
    if candidate.is_dir():
        dataset_root = find_bright_root(candidate)
        if dataset_root is not None:
            break
if dataset_root is None:
    modality_dirs = {}
    for candidate in dataset_candidates:
        if candidate.is_dir():
            print(f'[data] mounted dataset: {candidate}', flush=True)
            for modality in ('pre-event', 'post-event', 'target'):
                # BRIGHT uploads commonly contain modality/modality/*.tif; choose the innermost directory.
                matches = sorted((path for path in candidate.rglob(modality) if path.is_dir()), key=lambda path: (-len(path.parts), str(path)))
                if matches:
                    modality_dirs[modality] = matches[0]
    if set(modality_dirs) == {'pre-event', 'post-event', 'target'}:
        normalized = Path('/kaggle/working/bright-normalized')
        normalized.mkdir(parents=True, exist_ok=True)
        for modality, source in modality_dirs.items():
            link = normalized / modality
            if not link.exists():
                link.symlink_to(source, target_is_directory=True)
            print(f'[data] using {modality}: {source}', flush=True)
        dataset_root = normalized.resolve()
if dataset_root is None:
    archives = {}
    for candidate in dataset_candidates:
        if candidate.is_dir():
            for archive in candidate.rglob('*.zip'):
                for modality in ('pre-event', 'post-event', 'target'):
                    if modality in archive.stem.lower():
                        archives[modality] = archive
    if set(archives) == {'pre-event', 'post-event', 'target'}:
        extracted = Path('/kaggle/working/bright')
        extracted.mkdir(parents=True, exist_ok=True)
        for modality in ('pre-event', 'post-event', 'target'):
            print(f'[data] extracting official {archives[modality].name}', flush=True)
            with zipfile.ZipFile(archives[modality]) as archive:
                archive.extractall(extracted)
        dataset_root = find_bright_root(extracted)
        if dataset_root is None:
            modality_dirs = {}
            for modality in ('pre-event', 'post-event', 'target'):
                matches = sorted((path for path in extracted.rglob(modality) if path.is_dir()), key=lambda path: (-len(path.parts), str(path)))
                if matches:
                    modality_dirs[modality] = matches[0]
            if set(modality_dirs) == {'pre-event', 'post-event', 'target'}:
                normalized = Path('/kaggle/working/bright-normalized')
                normalized.mkdir(parents=True, exist_ok=True)
                for modality, source in modality_dirs.items():
                    link = normalized / modality
                    if not link.exists():
                        link.symlink_to(source, target_is_directory=True)
                    print(f'[data] using extracted {modality}: {source}', flush=True)
                dataset_root = normalized.resolve()
if dataset_root is None:
    raise RuntimeError('Attached BRIGHT dataset must contain pre-event, post-event, and target directories.')
os.environ['DISASTERLENS_BRIGHT_ROOT'] = str(dataset_root)
print(f'[Kaggle] repository: {repo_dir}', flush=True)
print(f'[Kaggle] official BRIGHT root: {dataset_root}', flush=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(repo_dir)], check=True)
cache_markers = sorted(Path('/kaggle/input').rglob('m1_cache.json')) if Path('/kaggle/input').is_dir() else []
if len(cache_markers) > 1:
    raise RuntimeError(f'Multiple M1 cache inputs found; attach only kushchaudhari/disaster-lens-m1-cache: {cache_markers}')
m1_cache_root = cache_markers[0].parent if cache_markers else None
if m1_cache_root is None:
    print('[M1 cache] no persistent cache attached; a full audit is required before training', flush=True)
else:
    print(f'[M1 cache] found persistent cache at {m1_cache_root}', flush=True)
    subprocess.run([sys.executable, '-u', 'scripts/restore_kaggle_m1_cache.py', '--cache-root', str(m1_cache_root), '--dataset-root', str(dataset_root), '--repo-root', str(repo_dir)], cwd=repo_dir, check=True)

In [ ]:
def run_step(title, command):
    print(f'\n{"=" * 72}\n{title}\n{"=" * 72}', flush=True)
    started = time.perf_counter()
    process = subprocess.Popen(command, cwd=repo_dir, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
    status = process.wait()
    if status:
        raise subprocess.CalledProcessError(status, command)
    print(f'[complete] {title} in {time.perf_counter() - started:.1f}s', flush=True)

manifest_path = repo_dir / 'data/manifests/bright_manifest.jsonl'
if not manifest_path.is_file():
    raise RuntimeError(
        'M3 cannot start because the validated M1 manifest is unavailable. '
        'Attach kushchaudhari/disaster-lens-m1-cache or set '
        'DISASTERLENS_FROM_MILESTONE=M1 for a one-time real-data audit.'
    )
events = sorted({json.loads(line)['event_id'] for line in manifest_path.read_text().splitlines() if line.strip()})
if not events:
    raise RuntimeError('The official BRIGHT manifest contains no events.')
print(f'[resume] validated M1 manifest is available for {len(events)} real events', flush=True)

In [ ]:
# Default to M3 only after the completed M2 run. Extend the pipeline only after
# reviewing its reports, via Kaggle environment variables if desired.
from_milestone = os.environ.get('DISASTERLENS_FROM_MILESTONE', 'M3').upper()
through_milestone = os.environ.get('DISASTERLENS_THROUGH_MILESTONE', 'M3').upper()
test_event = os.environ.get('DISASTERLENS_TEST_EVENT', events[0])
if test_event not in events:
    raise ValueError(f'Unknown TEST_EVENT {test_event!r}; choose one of the audited events, e.g. {events[:5]}')
command = [
    sys.executable, '-u', 'scripts/run_end_to_end.py',
    '--bright-root', str(dataset_root),
    '--from-milestone', from_milestone,
    '--through', through_milestone,
    '--test-event', test_event,
    '--m3-epochs', os.environ.get('DISASTERLENS_M3_EPOCHS', '60'),
    '--m4-epochs', os.environ.get('DISASTERLENS_M4_EPOCHS', '60'),
    '--monte-carlo-simulations', os.environ.get('DISASTERLENS_MONTE_CARLO_SIMULATIONS', '500'),
    '--sensitivity-samples', os.environ.get('DISASTERLENS_SENSITIVITY_SAMPLES', '1000'),
]
optional_arguments = {
    'DISASTERLENS_WORLDPOP': '--worldpop',
    'DISASTERLENS_WORLDPOP_SOURCE': '--worldpop-source',
    'DISASTERLENS_WORLDPOP_URL': '--worldpop-url',
    'DISASTERLENS_POPULATION_YEAR': '--population-year',
    'DISASTERLENS_POPULATION_SOURCE': '--population-source',
    'DISASTERLENS_POPULATION_VERSION': '--population-version',
    'DISASTERLENS_POPULATION_LICENSE': '--population-license',
    'DISASTERLENS_POPULATION_DOWNLOAD_DATE': '--population-download-date',
    'DISASTERLENS_ROADS': '--roads',
    'DISASTERLENS_FACILITIES': '--facilities',
    'DISASTERLENS_ROADS_SOURCE': '--roads-source',
    'DISASTERLENS_FACILITIES_SOURCE': '--facilities-source',
    'DISASTERLENS_OSM_BBOX': '--osm-bbox',
    'DISASTERLENS_OVERPASS_URL': '--overpass-url',
}
for variable, option in optional_arguments.items():
    value = os.environ.get(variable)
    if value:
        command.extend([option, value])
print(f'[pipeline] {from_milestone} through {through_milestone}; held-out event: {test_event}', flush=True)
run_step('[DisasterLens] resumable end-to-end milestone pipeline', command)

# Kaggle publishes files directly under /kaggle/working. Stage the durable
# result artifacts outside the cloned repository so `kaggle kernels output`
# can retrieve them after the run. Predictions are intentionally excluded:
# they can be very large and are reproducible from the saved checkpoints.
source_outputs = repo_dir / 'outputs'
if not source_outputs.is_dir():
    raise RuntimeError(f'Pipeline reported success but output directory is missing: {source_outputs}')
export_root = Path('/kaggle/working/disasterlens-results')
if export_root.exists():
    shutil.rmtree(export_root)
export_root.mkdir(parents=True)
for name in ('checkpoints', 'metrics', 'reports', 'logs', 'figures', 'maps', 'runs', 'calibration', 'geospatial', 'priority'):
    source = source_outputs / name
    if source.exists():
        shutil.copytree(source, export_root / name)
manifest_export = export_root / 'manifests'
manifest_export.mkdir(parents=True, exist_ok=True)
for name in ('bright_manifest.jsonl', 'bright_manifest.json', 'bright_normalization.json', 'audit_summary.json'):
    source = repo_dir / 'data' / 'manifests' / name
    if source.is_file():
        shutil.copy2(source, manifest_export / name)
split_source = repo_dir / 'data' / 'manifests' / 'splits'
if split_source.is_dir():
    shutil.copytree(split_source, manifest_export / 'splits')
(export_root / 'README.txt').write_text('Kaggle-exported DisasterLens artifacts. Generated from official attached BRIGHT data. Large prediction rasters are intentionally excluded; use checkpoints plus manifests to reproduce them.\n', encoding='utf-8')
print(f'[Kaggle] exported durable artifacts to {export_root}: ' + ', '.join(sorted(path.name for path in export_root.iterdir())), flush=True)